In [1]:
'''
AŞI VERİLERİ
total_vaccinations	Toplam yapılan doz sayısı (bir kişi 2 doz olduysa 2 sayılır)
people_vaccinated	En az 1 doz olan kişi sayısı
people_fully_vaccinated	Tam aşılı (2 doz vb.) kişi sayısı
total_boosters	Yapılan hatırlatma (booster) dozu sayısı
*_per_hundred uzantılı	Yüz kişi başına oran
'''

'\nAŞI VERİLERİ\ntotal_vaccinations\tToplam yapılan doz sayısı (bir kişi 2 doz olduysa 2 sayılır)\npeople_vaccinated\tEn az 1 doz olan kişi sayısı\npeople_fully_vaccinated\tTam aşılı (2 doz vb.) kişi sayısı\ntotal_boosters\tYapılan hatırlatma (booster) dozu sayısı\n*_per_hundred uzantılı\tYüz kişi başına oran\n'

In [2]:
#Aşılama verilerinin incelenmesi
vaccination_coverage = df_countries.groupby('location').agg(
    toplam_satir=('date', 'size'),
    ilk_doz_kaydi=('people_vaccinated_per_hundred', 'count'),
    tam_asi_kaydi=('people_fully_vaccinated_per_hundred', 'count')
)

vaccination_coverage['bos_ilk_doz_kaydi'] = (
    vaccination_coverage['toplam_satir']
    - vaccination_coverage['ilk_doz_kaydi']
)

vaccination_coverage['bos_tam_asi_kaydi'] = (
    vaccination_coverage['toplam_satir']
    - vaccination_coverage['tam_asi_kaydi']
)

print(
    vaccination_coverage
    .sort_values('tam_asi_kaydi')
    .head(15)
)

NameError: name 'df_countries' is not defined

In [ ]:
us_vaccination = df_countries.loc[
    df_countries['location'] == 'United States',
    ['date', 'people_vaccinated', 'people_fully_vaccinated']
].sort_values('date').copy()

us_vaccination['hafta_baslangici'] = (
    us_vaccination['date']
    - pd.to_timedelta(
        us_vaccination['date'].dt.dayofweek,
        unit='D'
    )
)

print(us_vaccination.head(10).to_string(index=False))

In [ ]:
usa_vaccination_start = pd.Series({
    'ilk_doz_ilk_kayit': us_vaccination.loc[
        us_vaccination['people_vaccinated'].notna(),
        'date'
    ].min(),

    'tam_asi_ilk_kayit': us_vaccination.loc[
        us_vaccination['people_fully_vaccinated'].notna(),
        'date'
    ].min()
})

print(usa_vaccination_start)

In [ ]:
usa_vaccination_start_period = us_vaccination.loc[
    us_vaccination['date'].between(
        '2020-12-13',
        '2021-01-10'
    ),
    ['date', 'people_vaccinated',
     'people_fully_vaccinated',
     'hafta_baslangici']
]

print(usa_vaccination_start_period.to_string(index=False))

In [ ]:
#ABD'de haftalık aşı nasıl değişmiş
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    us_vaccination['date'],
    us_vaccination['people_vaccinated'],
    label='En az bir doz alan kişi',
    color='steelblue',
    linewidth=2
)

ax.plot(
    us_vaccination['date'],
    us_vaccination['people_fully_vaccinated'],
    label='Tam aşılı kişi',
    color='seagreen',
    linewidth=2
)

ax.set_title('ABD: COVID-19 Aşılama İlerlemesi')
ax.set_xlabel('Tarih')
ax.set_ylabel('Kişi sayısı')

ax.ticklabel_format(axis='y', style='plain')
ax.grid(alpha=0.3)
ax.legend()

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
print('total_vaccinations' in df_countries.columns)

In [ ]:
#toplam doz nası değişmiş,en az bir doz olanların sayısı,tam doz olanlar.
us_vaccination = (
    df_countries.loc[
        df_countries['location'].eq('United States'),
        [
            'date',
            'people_vaccinated',
            'people_fully_vaccinated',
            'total_vaccinations'
        ]
    ]
    .sort_values('date')
    .copy()
)

fig, ax_kisiler = plt.subplots(figsize=(12, 6))

# Sol eksen: kişi sayıları
ax_kisiler.plot(
    us_vaccination['date'],
    us_vaccination['people_vaccinated'],
    color='steelblue',
    linewidth=2,
    label='En az bir doz alan kişi'
)

ax_kisiler.plot(
    us_vaccination['date'],
    us_vaccination['people_fully_vaccinated'],
    color='seagreen',
    linewidth=2,
    label='Tam aşılı kişi'
)

ax_kisiler.set_xlabel('Tarih')
ax_kisiler.set_ylabel('Kişi sayısı')
ax_kisiler.ticklabel_format(axis='y', style='plain')
ax_kisiler.grid(alpha=0.3)

# Sağ eksen: toplam uygulanan doz
ax_dozlar = ax_kisiler.twinx()

ax_dozlar.plot(
    us_vaccination['date'],
    us_vaccination['total_vaccinations'],
    color='darkorange',
    linewidth=2,
    label='Uygulanan toplam doz'
)

ax_dozlar.set_ylabel('Toplam uygulanan doz')
ax_dozlar.ticklabel_format(axis='y', style='plain')

# İki eksendeki açıklamaları tek yerde birleştir
cizgiler_1, etiketler_1 = ax_kisiler.get_legend_handles_labels()
cizgiler_2, etiketler_2 = ax_dozlar.get_legend_handles_labels()

ax_kisiler.legend(
    cizgiler_1 + cizgiler_2,
    etiketler_1 + etiketler_2,
    loc='upper left'
)

ax_kisiler.set_title('ABD: Aşılanan Kişiler ve Uygulanan Toplam Doz')
fig.autofmt_xdate()

plt.tight_layout()
plt.show()

In [ ]:
us_vaccination['hafta_baslangici'] = (
    us_vaccination['date']
    - pd.to_timedelta(
        us_vaccination['date'].dt.dayofweek,
        unit='D'
    )
)

weekly_us_vaccination = (
    us_vaccination
    .dropna(
        subset=[
            'people_vaccinated',
            'people_fully_vaccinated',
            'total_vaccinations'
        ]
    )
    .sort_values('date')
    .groupby('hafta_baslangici', as_index=False)
    .tail(1)
    [
        [
            'hafta_baslangici',
            'date',
            'people_vaccinated',
            'people_fully_vaccinated',
            'total_vaccinations'
        ]
    ]
    .sort_values('hafta_baslangici')
    .copy()
)

weekly_us_vaccination['ilk_doz_haftalik_artis'] = (
    weekly_us_vaccination['people_vaccinated'].diff()
)

weekly_us_vaccination['tam_asi_haftalik_artis'] = (
    weekly_us_vaccination['people_fully_vaccinated'].diff()
)

weekly_us_vaccination['toplam_doz_haftalik_artis'] = (
    weekly_us_vaccination['total_vaccinations'].diff()
)

print(weekly_us_vaccination.head(10).to_string(index=False))

In [ ]:
weekly_us_vaccination['onceki_kayit_tarihi'] = (
    weekly_us_vaccination['date'].shift(1)
)

weekly_us_vaccination['kayitlar_arasi_gun'] = (
    weekly_us_vaccination['date']
    - weekly_us_vaccination['onceki_kayit_tarihi']
).dt.days

week_gaps = weekly_us_vaccination.loc[
    weekly_us_vaccination['kayitlar_arasi_gun'] > 7,
    [
        'hafta_baslangici',
        'onceki_kayit_tarihi',
        'date',
        'kayitlar_arasi_gun'
    ]
]

print(week_gaps.to_string(index=False))

In [ ]:
plot_weekly_vaccination = weekly_us_vaccination.dropna(
    subset=[
        'ilk_doz_haftalik_artis',
        'tam_asi_haftalik_artis',
        'toplam_doz_haftalik_artis'
    ]
).copy()

fig, (ax_kisiler, ax_dozlar) = plt.subplots(
    2,
    1,
    figsize=(13, 8),
    sharex=True
)

# Haftalık yeni kişi artışları
ax_kisiler.plot(
    plot_weekly_vaccination['hafta_baslangici'],
    plot_weekly_vaccination['ilk_doz_haftalik_artis'],
    color='steelblue',
    label='En az bir doz alanlardaki haftalık artış'
)

ax_kisiler.plot(
    plot_weekly_vaccination['hafta_baslangici'],
    plot_weekly_vaccination['tam_asi_haftalik_artis'],
    color='seagreen',
    label='Tam aşılılardaki haftalık artış'
)

ax_kisiler.set_title('ABD: Haftalık Aşılama Artışı')
ax_kisiler.set_ylabel('Kişi sayısı')
ax_kisiler.ticklabel_format(axis='y', style='plain')
ax_kisiler.grid(alpha=0.3)
ax_kisiler.legend()

# Haftalık toplam doz artışı
ax_dozlar.plot(
    plot_weekly_vaccination['hafta_baslangici'],
    plot_weekly_vaccination['toplam_doz_haftalik_artis'],
    color='darkorange',
    label='Toplam uygulanan dozlardaki haftalık artış'
)

ax_dozlar.set_xlabel('Hafta başlangıcı')
ax_dozlar.set_ylabel('Doz sayısı')
ax_dozlar.ticklabel_format(axis='y', style='plain')
ax_dozlar.grid(alpha=0.3)
ax_dozlar.legend()

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
weekly_vaccination_comparison = (
    df_countries[
        [
            'location',
            'date',
            'people_vaccinated_per_hundred',
            'people_fully_vaccinated_per_hundred'
        ]
    ]
    .dropna(
        subset=[
            'people_vaccinated_per_hundred',
            'people_fully_vaccinated_per_hundred'
        ]
    )
    .copy()
)

weekly_vaccination_comparison['hafta_baslangici'] = (
    weekly_vaccination_comparison['date']
    - pd.to_timedelta(
        weekly_vaccination_comparison['date'].dt.dayofweek,
        unit='D'
    )
)

weekly_vaccination_comparison = (
    weekly_vaccination_comparison
    .sort_values('date')
    .groupby(['location', 'hafta_baslangici'], as_index=False)
    .tail(1)
    .sort_values(['location', 'hafta_baslangici'])
    .copy()
)

print(weekly_vaccination_comparison.head(10).to_string(index=False))

In [ ]:
vaccination_week_coverage = (
    weekly_vaccination_comparison
    .groupby('hafta_baslangici')['location']
    .nunique()
    .sort_values(ascending=False)
)

print(vaccination_week_coverage.head(10))

In [ ]:
selected_week = pd.Timestamp('2021-09-06')

selected_vaccination_week = (
    weekly_vaccination_comparison.loc[
        weekly_vaccination_comparison['hafta_baslangici'].eq(selected_week),
        [
            'location',
            'people_vaccinated_per_hundred',
            'people_fully_vaccinated_per_hundred'
        ]
    ]
    .sort_values(
        'people_fully_vaccinated_per_hundred',
        ascending=False
    )
    .head(15)
    .sort_values('people_fully_vaccinated_per_hundred')
)

print(selected_vaccination_week.to_string(index=False))

In [ ]:
y = np.arange(len(selected_vaccination_week))
bar_height = 0.38

fig, ax = plt.subplots(figsize=(11, 8))

ax.barh(
    y - bar_height / 2,
    selected_vaccination_week['people_vaccinated_per_hundred'],
    height=bar_height,
    color='steelblue',
    label='En az bir doz alanlar'
)

ax.barh(
    y + bar_height / 2,
    selected_vaccination_week['people_fully_vaccinated_per_hundred'],
    height=bar_height,
    color='seagreen',
    label='Tam aşılılar'
)

ax.set_yticks(y)
ax.set_yticklabels(selected_vaccination_week['location'])

ax.set_title('6 Eylül 2021 Haftası: Aşı Kapsamı Karşılaştırması')
ax.set_xlabel('100 kişi başına kişi sayısı')
ax.set_ylabel('Ülke / bölge')

ax.set_xlim(0, 100)
ax.grid(axis='x', alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
#Hastane yatakları kapasitesi,hastanede yatan hasta sayısı,o gün yoğun bakımda olanlar
hospital_columns = [
    'location',
    'date',
    'hospital_beds_per_thousand',
    'hosp_patients',
    'icu_patients'
]

print(df_countries[hospital_columns].info())

In [ ]:
#hospital_beds_per_thousand:1000 kişi başına hastane yatak kapasitesi
#hosp_patients:O gün hastanede yatan COVID-19 hasta sayısı.
#icu_patients:O gün yoğun bakımda bulunan COVID-19 hasta sayısı.
bed_check = df_countries.groupby('location').agg(
    toplam_satir=('date', 'size'),
    dolu_yatak_kaydi=('hospital_beds_per_thousand', 'count'),
    farkli_yatak_degeri=('hospital_beds_per_thousand', 'nunique'),
    en_dusuk_yatak=('hospital_beds_per_thousand', 'min'),
    en_yuksek_yatak=('hospital_beds_per_thousand', 'max')
)

print(
    bed_check
    .sort_values('dolu_yatak_kaydi')
    .head(15)
)

In [ ]:
booster_coverage = (
    df_countries
    .groupby('location')
    .agg(
        toplam_satir=('date', 'size'),
        dolu_hatirlatma_dozu_kaydi=('total_boosters', 'count'),
        dolu_hatirlatma_orani_kaydi=(
            'total_boosters_per_hundred',
            'count'
        )
    )
)

valid_booster_range = (
    df_countries[
        df_countries['total_boosters'].notna()
    ]
    .groupby('location')
    .agg(
        ilk_hatirlatma_dozu_tarihi=('date', 'min'),
        son_hatirlatma_dozu_tarihi=('date', 'max')
    )
)

booster_coverage = booster_coverage.join(valid_booster_range)

booster_coverage['bos_hatirlatma_dozu_kaydi'] = (
    booster_coverage['toplam_satir']
    - booster_coverage['dolu_hatirlatma_dozu_kaydi']
)

print(
    booster_coverage
    .sort_values('dolu_hatirlatma_dozu_kaydi', ascending=False)
    .head(20)
)

In [ ]:
us_booster_data = (
    df_countries[
        df_countries['location'] == 'United States'
    ][
        [
            'date',
            'total_boosters',
            'total_boosters_per_hundred',
            'people_fully_vaccinated'
        ]
    ]
    .sort_values('date')
)

us_booster_valid = us_booster_data[
    us_booster_data['total_boosters'].notna()
].copy()

expected_dates = pd.date_range(
    us_booster_valid['date'].min(),
    us_booster_valid['date'].max(),
    freq='D'
)

missing_booster_dates = expected_dates.difference(
    us_booster_valid['date']
)

print('İlk hatırlatma dozu kaydı:', us_booster_valid['date'].min())
print('Son hatırlatma dozu kaydı:', us_booster_valid['date'].max())
print('Hatırlatma dozu bulunan gün:', len(us_booster_valid))
print('Aradaki eksik gün sayısı:', len(missing_booster_dates))
print(missing_booster_dates[:10])

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(
    us_booster_valid['date'],
    us_booster_valid['people_fully_vaccinated'],
    color='seagreen',
    linewidth=2,
    label='Tam aşılı kişi sayısı'
)

ax.plot(
    us_booster_valid['date'],
    us_booster_valid['total_boosters'],
    color='darkorange',
    linewidth=2,
    label='Uygulanan toplam hatırlatma dozu'
)

ax.set_title(
    'ABD: Tam Aşılama ve Hatırlatma Dozu İlerlemesi'
)
ax.set_xlabel('Tarih')
ax.set_ylabel('Kişi / uygulanan doz sayısı')
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
us_booster_weekly = (
    us_booster_valid
    .assign(
        hafta_baslangici=(
            us_booster_valid['date']
            - pd.to_timedelta(
                us_booster_valid['date'].dt.dayofweek,
                unit='D'
            )
        )
    )
    .sort_values('date')
    .groupby('hafta_baslangici', as_index=False)
    .last()
)

us_booster_weekly['haftalik_hatirlatma_dozu_artisi'] = (
    us_booster_weekly['total_boosters']
    .diff()
)

print(
    us_booster_weekly[
        [
            'hafta_baslangici',
            'total_boosters',
            'haftalik_hatirlatma_dozu_artisi'
        ]
    ]
    .head(10)
)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    us_booster_weekly['hafta_baslangici'],
    us_booster_weekly['haftalik_hatirlatma_dozu_artisi'],
    color='darkorange',
    linewidth=1.8
)

ax.set_title('ABD: Haftalık Hatırlatma Dozu Artışı')
ax.set_xlabel('Hafta başlangıcı')
ax.set_ylabel('Haftalık uygulanan hatırlatma dozu')

ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()